# Healthcare SLM Fine-Tuning: Google Gemma 3 (270M)
### Clinical Notes to Structured SOAP Summarizer

This notebook fine-tunes **`google/gemma-3-270m-it`** using **PEFT (LoRA)** and **TRL (`SFTTrainer`)** on clinical doctor notes to generate standardized **SOAP medical reports** (**S**ubjective, **O**bjective, **A**ssessment, **P**lan).

**Dataset:** [`AmareshHebbar/clinical-summarizer-sft`](https://huggingface.co/datasets/AmareshHebbar/clinical-summarizer-sft) (AxisMapper Medical AI Suite)
**Hardware:** Designed to run via PyCharm connected to Google Colab (T4 / A100 GPU).

In [ ]:
# Cell 1: Environment & GPU Verification
!pip install -q transformers peft trl datasets accelerate bitsandbytes

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Cell 2: Load Gemma 3 (270M) & Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-3-270m-it"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto"
)

print(f"Base model loaded: {MODEL_ID}")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 3: Attach PEFT / LoRA Adapters
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 4: Load & Format Healthcare Dataset
from datasets import load_dataset

# Load 1,500 samples for a fast, focused fine-tuning demo run
raw_dataset = load_dataset("AmareshHebbar/clinical-summarizer-sft", split="train[:1500]")

def format_clinical_chat(example):
    formatted_messages = []
    system_text = ""
    
    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            system_text = content
        elif role == "user":
            prompt = f"{system_text}\n\n{content}".strip() if system_text else content
            system_text = ""
            formatted_messages.append({"role": "user", "content": prompt})
        elif role in ("assistant", "model"):
            formatted_messages.append({"role": "model", "content": content})
            
    text = tokenizer.apply_chat_template(formatted_messages, tokenize=False)
    return {"text": text}

train_dataset = raw_dataset.map(format_clinical_chat)
print("Sample Formatted Clinical Entry:")
print(train_dataset[0]["text"][:600] + "...")

In [ ]:
# Cell 5: Supervised Fine-Tuning (SFTTrainer)
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./gemma3-clinical-soap",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=60,                 # Fast demo run (~2 mins on Colab T4)
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting Healthcare SLM Fine-Tuning...")
trainer.train()

In [ ]:
# Cell 6: Test Clinical Note -> SOAP Synthesis
sample_doctor_note = (
    "You are a clinical summarization assistant.\n\n"
    "Summarize the following clinical note into a structured SOAP note:\n"
    "Pt 62yo F presenting with 4-day history of progressively worsening shortness of breath and mild chest tightness on exertion. "
    "No fever, no hemoptysis. History of CAD and mild asthma. Meds: Albuterol PRN, Atorvastatin 20mg. "
    "Vitals: BP 142/86, HR 88, RR 20, SpO2 94% on RA. Lungs show faint wheezing at right base, heart regular rhythm."
)

messages = [{"role": "user", "content": sample_doctor_note}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.3,
        top_p=0.9,
        do_sample=True
    )

generated_soap = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("Generated SOAP Clinical Summary:")
print(generated_soap)

In [ ]:
# Cell 7: Save & Export LoRA Adapter
OUTPUT_DIR = "./gemma3_clinical_soap_adapter"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Clinical SOAP LoRA adapter successfully saved to {OUTPUT_DIR}")